# EDA de INEGI-POP-001: población por localidad (ITER 2020)

Pregunta del proyecto: ¿Dónde se concentran las poblaciones de Sonora con menor accesibilidad geográfica a servicios de salud?

Qué aporta el ITER a esa pregunta:
- Dónde vive la gente: cada localidad con sus coordenadas y su población total.
- Qué tan vulnerable es: personas de 60 años y más, con discapacidad, sin afiliación a servicios de salud o que hablan lengua indígena, y viviendas sin auto ni moto.

Este notebook revisa solo lo que sirve a esas dos cosas y a la limpieza de sus columnas. Es de solo lectura: el zip se lee directo y no se cambia ningún dato.

- Fuente: INEGI, Censo de Población y Vivienda 2020, Principales resultados por localidad (ITER)
- Página: https://www.inegi.org.mx/app/scitel/Default?ev=9
- Descarga: https://www.inegi.org.mx/contenidos/programas/ccpv/iter/zip/iter2020/iter_26csv20.zip
- Fecha de descarga: 11/10/2026
- Descriptor: https://www.inegi.org.mx/app/scitel/doc/descriptor/fd_iter_cpv2020.pdf (cuarta edición, 2022)

## 0. Preparación

Carga de librerías, rutas del proyecto y configuración de las gráficas.

In [1]:
import hashlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from health_data.pipeline import RAW_DATA_DIR  # noqa: E402

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)
pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"

Huella digital del zip (SHA-256). Cambia si el archivo cambia, así que confirma que es el mismo que se descargó.

In [2]:
ARCHIVO = RAW_DATA_DIR / "iter_26csv20.zip"
hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()

'3f3795a7963d9ccdba0b19f61f7b70326ee3a48ef5af6b6b65486ff53542c366'

Lectura del archivo que viene dentro del zip. Todo se lee como texto y nada se convierte en vacío, para conservar los ceros a la izquierda de las claves y las marcas "*" y "N/D" tal como vienen.

In [3]:
df = pd.read_csv(ARCHIVO, dtype=str, encoding="utf-8-sig", keep_default_na=False)
df.shape

(7500, 286)

Colores y funciones del notebook. Una función convierte las columnas a número y deja vacías las marcas especiales.

In [4]:
AZUL, NARANJA = "#2a78d6", "#eb6834"
AZULES = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]
ESPECIALES = ["*", "N/D", "N/A", ""]
CENTRO_SONORA = {"lat": 29.3, "lon": -110.6}


def a_numero(serie):
    return pd.to_numeric(serie.str.strip().where(~serie.isin(ESPECIALES)))


def localidades_y_poblacion(tabla, titulo, eje_x, etiquetas=True):
    largo = (
        tabla[["% localidades", "% población"]]
        .rename_axis(eje_x)
        .reset_index()
        .melt(id_vars=eje_x, var_name="Medida", value_name="Porcentaje")
    )
    fig = px.bar(
        largo,
        x=eje_x,
        y="Porcentaje",
        color="Medida",
        barmode="group",
        text="Porcentaje" if etiquetas else None,
        color_discrete_map={"% localidades": AZUL, "% población": NARANJA},
        title=titulo,
    )
    if etiquetas:
        fig.update_traces(texttemplate="%{text:.1f}", textposition="outside", cliponaxis=False)
    fig.update_layout(
        height=420, yaxis_title="Porcentaje", legend_title=None, bargroupgap=0.05, barcornerradius=4
    )
    fig.show()

## 1. Estructura

Cómo vienen el archivo y sus claves. Usa todas las filas del archivo.

Primeras filas, tal como vienen.

In [5]:
df.iloc[:6, :10]

,ENTIDAD,NOM_ENT,MUN,NOM_MUN,LOC,NOM_LOC,LONGITUD,LATITUD,ALTITUD,POBTOT
0,26,Sonora,000,Total de la entidad Sonora,0000,Total de la Entidad,,,,2944840
1,26,Sonora,000,Total de la entidad Sonora,9998,Localidades de una vivienda,,,,16883
2,26,Sonora,000,Total de la entidad Sonora,9999,Localidades de dos viviendas,,,,5087
3,26,Sonora,001,Aconchi,0000,Total del Municipio,,,,2563
4,26,Sonora,001,Aconchi,0001,Aconchi,"110°13'42.787"" W","29°49'33.141"" N",0607,1650
5,26,Sonora,001,Aconchi,0003,La Estancia,"110°12'40.335"" W","29°47'43.735"" N",0604,761


Número de caracteres de las claves, las coordenadas y el tamaño de localidad.

In [6]:
pd.DataFrame(
    {c: df[c].str.len().value_counts() for c in ["ENTIDAD", "MUN", "LOC", "LATITUD", "LONGITUD", "TAMLOC"]}
).fillna(0).astype(int).rename_axis("caracteres")

,ENTIDAD,MUN,LOC,LATITUD,LONGITUD,TAMLOC
caracteres,,,,,,
0,0,0,0,200,200,0
1,0,0,0,0,0,7491
2,7500,0,0,0,0,9
3,0,7500,0,0,0,0
4,0,0,7500,0,0,0
15,0,0,0,7300,0,0
16,0,0,0,0,7300,0


Resumen:
- El archivo tiene 7,500 filas y 286 columnas.
- Todo viene como texto, y las claves conservan sus ceros a la izquierda.
- Las coordenadas vienen en grados, minutos y segundos. 200 filas no las traen.
- El tamaño de localidad viene sin el cero a la izquierda que marca el descriptor (1 en lugar de 01).

## 2. Columnas que se conservan

Las columnas que sirven para ubicar a la población y medir su vulnerabilidad, con su tipo y su denominador. Usa todas las filas del archivo.

El denominador es el total entre el que se divide un conteo para volverlo porcentaje. Sale del universo que da el descriptor en cada definición; por ejemplo, en lengua indígena el universo son las personas de 3 años y más. La última columna cuenta las filas en las que un conteo pasa a su denominador; si el denominador es correcto, debe ser 0.

In [7]:
CONSERVAR = pd.DataFrame(
    [
        ("ENTIDAD", "Clave de la entidad", "identificador", "llave", None),
        ("MUN", "Clave del municipio", "identificador", "llave", None),
        ("LOC", "Clave de la localidad", "identificador", "llave", None),
        ("NOM_MUN", "Nombre del municipio", "texto", "nombre", None),
        ("NOM_LOC", "Nombre de la localidad", "texto", "nombre", None),
        ("LATITUD", "Latitud", "geográfica", "ubicación", None),
        ("LONGITUD", "Longitud", "geográfica", "ubicación", None),
        ("TAMLOC", "Tamaño de localidad", "categórica ordinal", "tamaño", None),
        ("POBTOT", "Población total", "conteo de personas", "población y denominador", None),
        ("P_3YMAS", "Población de 3 años y más", "conteo de personas", "denominador", "POBTOT"),
        ("P_60YMAS", "Población de 60 años y más", "conteo de personas", "indicador de vulnerabilidad", "POBTOT"),
        ("PCON_DISC", "Población con discapacidad", "conteo de personas", "indicador de vulnerabilidad", "POBTOT"),
        ("PSINDER", "Población sin afiliación a servicios de salud", "conteo de personas", "indicador de vulnerabilidad", "POBTOT"),
        ("P3YM_HLI", "Población de 3 años y más que habla lengua indígena", "conteo de personas", "indicador de vulnerabilidad", "P_3YMAS"),
        ("TVIVHAB", "Viviendas habitadas", "conteo de viviendas", "confidencialidad y vivienda colectiva", None),
        ("VIVPARH_CV", "Viviendas particulares habitadas con características", "conteo de viviendas", "denominador", "TVIVHAB"),
        ("VPH_NDACMM", "Viviendas sin auto ni moto", "conteo de viviendas", "indicador de vulnerabilidad", "VIVPARH_CV"),
    ],
    columns=["columna", "descripción", "tipo", "uso", "denominador"],
).set_index("columna")
INDICADORES = CONSERVAR.loc[CONSERVAR["uso"].eq("indicador de vulnerabilidad"), "denominador"].to_dict()
CONTEOS = CONSERVAR.index[CONSERVAR["tipo"].str.startswith("conteo")].tolist()
NOMBRES = {
    "P_60YMAS": "60 años y más",
    "PCON_DISC": "Con discapacidad",
    "PSINDER": "Sin afiliación a servicios de salud",
    "P3YM_HLI": "Habla lengua indígena",
    "VPH_NDACMM": "Viviendas sin auto ni moto",
}

numeros = df[CONTEOS].apply(a_numero)
CONSERVAR["filas mayores que el denominador"] = [
    numeros[c].gt(numeros[d]).sum() if d else "" for c, d in CONSERVAR["denominador"].items()
]
print(f"Columnas que se conservan: {len(CONSERVAR)} de {df.shape[1]}")
CONSERVAR.fillna("")

Columnas que se conservan: 17 de 286


,descripción,tipo,uso,denominador,filas mayores que el denominador
columna,,,,,
ENTIDAD,Clave de la entidad,identificador,llave,,
MUN,Clave del municipio,identificador,llave,,
LOC,Clave de la localidad,identificador,llave,,
NOM_MUN,Nombre del municipio,texto,nombre,,
NOM_LOC,Nombre de la localidad,texto,nombre,,
LATITUD,Latitud,geográfica,ubicación,,
LONGITUD,Longitud,geográfica,ubicación,,
TAMLOC,Tamaño de localidad,categórica ordinal,tamaño,,
POBTOT,Población total,conteo de personas,población y denominador,,


Resumen:
- Se conservan 17 de las 286 columnas.
- Cinco conteos son los indicadores de vulnerabilidad; las demás columnas sirven de clave, nombre, ubicación o denominador.
- Ningún conteo pasa a su denominador en ninguna fila.
- Un conteo solo se compara entre localidades después de dividirlo entre su denominador.

## 3. Tipos de fila

Qué filas son localidades y cuáles son totales o acumulados. Usa todas las filas del archivo.

Según el descriptor, la clave de localidad 0000 marca un total: del estado si la clave de municipio es 000 y del municipio si no. Las claves 9998 y 9999 marcan acumulados, es decir, la suma de las localidades de una y de dos viviendas.

In [8]:
def tipo_de_fila(clave_mun, clave_loc):
    if clave_loc == "0000":
        return "total estatal" if clave_mun == "000" else "total municipal"
    if clave_loc == "9998":
        return "acumulado de 1 vivienda"
    if clave_loc == "9999":
        return "acumulado de 2 viviendas"
    return "localidad"


tipo = pd.Series([tipo_de_fila(m, l) for m, l in zip(df["MUN"], df["LOC"])], index=df.index)
nivel = df["MUN"].eq("000").map({True: "estatal", False: "municipal"})
print(f"Filas de totales o acumulados: {tipo.ne('localidad').sum()}")
pd.crosstab(tipo, nivel, margins=True, margins_name="Total").rename_axis(index="tipo de fila", columns="nivel")

Filas de totales o acumulados: 200


nivel,estatal,municipal,Total
tipo de fila,,,
acumulado de 1 vivienda,1,67,68
acumulado de 2 viviendas,1,58,59
localidad,0,7300,7300
total estatal,1,0,1
total municipal,0,72,72
Total,3,7497,7500


Separación de las localidades, con su clave de 9 dígitos (entidad, municipio y localidad) y algunas columnas como número.

In [9]:
cvegeo = df["ENTIDAD"] + df["MUN"] + df["LOC"]
pobtot = a_numero(df["POBTOT"])

locs = df[tipo == "localidad"]
pob = pobtot[locs.index]
tvivhab = a_numero(locs["TVIVHAB"])
len(locs)

7300

Población total por municipio: total municipal contra la suma de sus localidades, con y sin los acumulados.

In [10]:
mun = (
    pd.DataFrame({"MUN": df["MUN"], "tipo": tipo, "POBTOT": pobtot})
    .query("MUN != '000'")
    .pivot_table(index="MUN", columns="tipo", values="POBTOT", aggfunc="sum", fill_value=0)
)
acumulados = mun["acumulado de 1 vivienda"] + mun["acumulado de 2 viviendas"]
pd.Series(
    {
        "municipios": len(mun),
        "municipios sin acumulados": acumulados.eq(0).sum(),
        "total = localidades": mun["total municipal"].eq(mun["localidad"]).sum(),
        "total = localidades + acumulados": mun["total municipal"].eq(mun["localidad"] + acumulados).sum(),
    }
)

municipios                          72
municipios sin acumulados            4
total = localidades                 72
total = localidades + acumulados     4
dtype: int64

Lo mismo para el estado.

In [11]:
pd.Series(
    {
        "total estatal": pobtot[tipo.eq("total estatal")].sum(),
        "suma de totales municipales": pobtot[tipo.eq("total municipal")].sum(),
        "suma de localidades": pob.sum(),
        "acumulados estatales": pobtot[df["MUN"].eq("000") & tipo.str.startswith("acumulado")].sum(),
    }
)

total estatal                  2944840
suma de totales municipales    2944840
suma de localidades            2944840
acumulados estatales             21970
dtype: int64

Localidades con una y dos viviendas habitadas contra los acumulados municipales.

In [12]:
pd.DataFrame(
    {
        "localidades": [tvivhab.eq(1).sum(), tvivhab.eq(2).sum()],
        "población de las localidades": [pob[tvivhab.eq(1)].sum(), pob[tvivhab.eq(2)].sum()],
        "población de los acumulados": [mun["acumulado de 1 vivienda"].sum(), mun["acumulado de 2 viviendas"].sum()],
    },
    index=["1 vivienda", "2 viviendas"],
)

,localidades,población de las localidades,población de los acumulados
1 vivienda,4578,16883,16883
2 viviendas,838,5087,5087


Prueba con un indicador que no se publica para esas localidades (60 años y más): total municipal menos localidades menos acumulados. Si los acumulados resumen a esas localidades, la diferencia es 0. Se muestran solo los municipios donde no lo es.

In [13]:
p60 = (
    pd.DataFrame({"MUN": df["MUN"], "tipo": tipo, "P_60YMAS": a_numero(df["P_60YMAS"])})
    .query("MUN != '000'")
    .pivot_table(index="MUN", columns="tipo", values="P_60YMAS", aggfunc="sum", fill_value=0)
)
diferencia = (
    p60["total municipal"] - p60["localidad"] - p60["acumulado de 1 vivienda"] - p60["acumulado de 2 viviendas"]
)
acumulado_reservado = df.loc[tipo.str.startswith("acumulado") & df["P_60YMAS"].eq("*"), "MUN"]
print(f"Municipios con diferencia 0: {diferencia.eq(0).sum()} de {len(diferencia)}")
pd.DataFrame(
    {"diferencia": diferencia, "acumulado reservado (*)": diferencia.index.isin(acumulado_reservado)}
)[diferencia.ne(0)]

Municipios con diferencia 0: 61 de 72


,diferencia,acumulado reservado (*)
MUN,,
007,2.0,True
010,4.0,True
014,3.0,True
023,5.0,True
030,1.0,False
032,1.0,True
034,2.0,True
037,1.0,True
044,1.0,True


Resumen:
- Hay 7,300 localidades y 200 filas de totales o acumulados.
- En los 72 municipios, la suma de la población de las localidades es igual al total municipal.
- Al sumar también los acumulados ya no cuadra, salvo en los 4 municipios que no tienen acumulados.
- En el estado, el total, los municipios y las localidades suman 2,944,840 personas.
- Las 4,578 localidades de una vivienda suman lo mismo que su acumulado: 16,883 personas.
- Las 838 localidades de dos viviendas también: 5,087 personas.
- Con 60 años y más, 61 de los 72 municipios no tienen diferencia.
- En 10 de los otros el acumulado está reservado y su valor no se conoce. En Hermosillo la diferencia es de 1 persona, sin causa clara.
- Conclusión: los acumulados repiten localidades, así que quitarlos no pierde ninguna. Aun así, son el único lugar donde aparecen los indicadores de las localidades de una y dos viviendas, sumados por municipio.

## 4. Claves

Revisión de la clave de entidad, de los municipios y de la clave de 9 dígitos. Usa todas las filas del archivo.

In [14]:
pd.Series(
    {
        "valores de ENTIDAD": ", ".join(df["ENTIDAD"].unique()),
        "municipios con localidades": locs["MUN"].nunique(),
        "municipios con total municipal": df.loc[tipo.eq("total municipal"), "MUN"].nunique(),
        "claves de municipio del 001 al 072": set(locs["MUN"]) == {f"{i:03d}" for i in range(1, 73)},
        "un nombre por clave de municipio": locs.groupby("MUN")["NOM_MUN"].nunique().eq(1).all(),
        "claves de 9 dígitos": cvegeo.str.fullmatch(r"\d{9}").sum(),
        "claves repetidas": cvegeo.duplicated().sum(),
    }
)

valores de ENTIDAD                      26
municipios con localidades              72
municipios con total municipal          72
claves de municipio del 001 al 072    True
un nombre por clave de municipio      True
claves de 9 dígitos                   7500
claves repetidas                         0
dtype: object

Resumen:
- La entidad siempre es 26 y hay 72 municipios, cada uno con un solo nombre.
- Las 7,500 claves de 9 dígitos son únicas, así que la clave sirve como llave.

## 5. Valores especiales

Dónde faltan datos en las columnas que se conservan y cuánta población tiene todos los indicadores. Usa las 7,300 localidades.

El archivo usa dos marcas: "*" cuando el dato se reserva por confidencialidad y "N/D" cuando no está disponible. Conteo de cada marca, y de otras marcas o vacíos, en las columnas que se conservan.

In [15]:
pd.DataFrame(
    {nombre: locs[CONSERVAR.index].eq(v).sum() for v, nombre in {"*": "*", "N/D": "N/D", "N/A": "N/A", "": "(vacío)"}.items()}
)

,*,N/D,N/A,(vacío)
ENTIDAD,0,0,0,0
MUN,0,0,0,0
LOC,0,0,0,0
NOM_MUN,0,0,0,0
NOM_LOC,0,0,0,0
LATITUD,0,0,0,0
LONGITUD,0,0,0,0
TAMLOC,0,0,0,0
POBTOT,0,0,0,0
P_3YMAS,5416,31,0,0


Clasificación de las localidades según la marca de sus indicadores, y cruce con el número de viviendas habitadas. Antes se revisa que los cinco indicadores y sus denominadores tengan la misma marca en cada localidad.

In [16]:
COLUMNAS_INDICADORES = list(INDICADORES) + ["P_3YMAS", "VIVPARH_CV"]
marca = locs[COLUMNAS_INDICADORES].where(locs[COLUMNAS_INDICADORES].isin(["*", "N/D"]), "dato")
print(f"Localidades con marcas distintas entre sus indicadores: {marca.nunique(axis=1).gt(1).sum()}")
disponibilidad = (
    marca["P_60YMAS"]
    .map({"*": "confidencial (*)", "N/D": "no disponible (N/D)", "dato": "indicadores completos"})
    .rename("disponibilidad")
)
pd.crosstab(
    disponibilidad,
    tvivhab.clip(upper=3).map({1: "1", 2: "2", 3: "3 o más"}),
    margins=True,
    margins_name="Total",
).rename_axis(columns="viviendas habitadas")

Localidades con marcas distintas entre sus indicadores: 0


viviendas habitadas,1,2,3 o más,Total
disponibilidad,,,,
confidencial (*),4578,838,0,5416
indicadores completos,0,0,1853,1853
no disponible (N/D),0,0,31,31
Total,4578,838,1884,7300


Hallazgo: 5,416 localidades solo traen población y viviendas; 1,853 tienen todos los indicadores.

Localidades y población según la disponibilidad de indicadores.

In [17]:
resumen = pd.DataFrame({"localidades": disponibilidad.value_counts(), "población": pob.groupby(disponibilidad).sum()})
resumen["% localidades"] = (resumen["localidades"] / resumen["localidades"].sum() * 100).round(2)
resumen["% población"] = (resumen["población"] / resumen["población"].sum() * 100).round(2)
resumen

,localidades,población,% localidades,% población
disponibilidad,,,,
confidencial (*),5416,21970,74.19,0.75
indicadores completos,1853,2922212,25.38,99.23
no disponible (N/D),31,658,0.42,0.02


In [18]:
localidades_y_poblacion(resumen, "Localidades y población según disponibilidad de indicadores", "Disponibilidad")

Localidades no disponibles (N/D) por municipio.

In [19]:
(
    locs[disponibilidad.eq("no disponible (N/D)")]
    .assign(POBTOT=pob)
    .groupby("NOM_MUN")
    .agg(localidades=("LOC", "size"), poblacion=("POBTOT", "sum"))
)

,localidades,poblacion
NOM_MUN,,
Guaymas,1,13
Hermosillo,30,645


Resumen:
- Las localidades confidenciales son justo las de una o dos viviendas habitadas: el censo no publica sus indicadores para proteger a quienes viven ahí.
- Son el 74.19 % de las localidades, pero reúnen solo el 0.75 % de la población.
- Las 1,853 localidades con todos los indicadores reúnen el 99.23 % de la población.
- Hay 31 localidades no disponibles, con 658 personas. 30 están en Hermosillo y 1 en Guaymas.
- Según el descriptor, las no disponibles son lugares visitados donde no se pudieron recoger datos; solo hay una estimación de su población y de sus viviendas.
- En cada localidad, los cinco indicadores tienen la misma marca. Las columnas que se conservan no tienen vacíos ni otras marcas.

## 6. Coordenadas

Conversión de las coordenadas a grados decimales, revisión de ubicaciones y mapa de las localidades con y sin indicadores. Usa las 7,300 localidades.

Según el descriptor, las coordenadas vienen en grados, minutos y segundos con milésimas, más la letra del hemisferio; por ejemplo, 29°49'33.141" N. Conteo de localidades que cumplen ese formato.

In [20]:
PATRON = r"""(\d{1,3})°(\d{2})'(\d{2}\.\d{3})" ([NSEW])"""


def a_decimal(serie):
    partes = serie.str.extract(f"^{PATRON}$")
    grados = partes[0].astype(float) + partes[1].astype(float) / 60 + partes[2].astype(float) / 3600
    return grados.where(partes[3].isin(["N", "E"]), -grados)


pd.DataFrame(
    {
        c: {
            "cumplen el formato": locs[c].str.fullmatch(PATRON).sum(),
            "minutos o segundos de 60 o más": locs[c].str.extract(PATRON)[[1, 2]].astype(float).ge(60).any(axis=1).sum(),
            "hemisferio": ", ".join(locs[c].str[-1].unique()),
        }
        for c in ["LATITUD", "LONGITUD"]
    }
)

,LATITUD,LONGITUD
cumplen el formato,7300,7300
minutos o segundos de 60 o más,0,0
hemisferio,N,W


Conversión a grados decimales, con la longitud negativa por estar al oeste, y revisión con una caja que encierra a Sonora. La caja va de 26.0 a 32.6 grados de latitud. En longitud va de −115.2 a −108.3 grados. Es la misma caja que se usó en el EDA de SALUD-001 y no es el límite real del estado.

In [21]:
lat, lon = a_decimal(locs["LATITUD"]), a_decimal(locs["LONGITUD"])

LAT_MIN, LAT_MAX = 26.0, 32.6
LON_MIN, LON_MAX = -115.2, -108.3
coord = pd.Series("dentro de la caja", index=locs.index)
coord[~(lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX))] = "fuera de la caja"
coord[lat.isna() | lon.isna()] = "sin coordenadas"
coord.value_counts()

dentro de la caja    7300
Name: count, dtype: int64

Localidades con coordenadas repetidas y filas de totales o acumulados sin coordenadas.

In [22]:
pd.Series(
    {
        "localidades con coordenadas repetidas": pd.concat([lat, lon], axis=1).duplicated(keep=False).sum(),
        "filas de totales o acumulados sin coordenadas": df.loc[tipo.ne("localidad"), ["LATITUD", "LONGITUD"]].eq("").all(axis=1).sum(),
    }
)

localidades con coordenadas repetidas              0
filas de totales o acumulados sin coordenadas    200
dtype: int64

Mapa de las localidades: en azul las que tienen indicadores y en naranja las que no. El tamaño del punto crece con la población. El cursor muestra la clave y el municipio; el nombre no se muestra porque algunos son nombres de personas.

In [23]:
mapa = pd.DataFrame(
    {
        "etiqueta": cvegeo[locs.index] + " · " + locs["NOM_MUN"],
        "Indicadores": disponibilidad.eq("indicadores completos").map({True: "con indicadores", False: "sin indicadores"}),
        "Población": pob,
        "lat": lat.round(5),
        "lon": lon.round(5),
    }
)
print(mapa["Indicadores"].value_counts().to_dict())
fig = px.scatter_map(
    mapa,
    lat="lat",
    lon="lon",
    size="Población",
    size_max=40,
    color="Indicadores",
    category_orders={"Indicadores": ["sin indicadores", "con indicadores"]},
    color_discrete_map={"con indicadores": AZUL, "sin indicadores": NARANJA},
    hover_name="etiqueta",
    center=CENTRO_SONORA,
    zoom=5.2,
    map_style="carto-positron",
    height=650,
    title="Localidades con y sin indicadores de vulnerabilidad",
)
fig.update_traces(
    marker_sizemin=3,
    marker_opacity=0.7,
    hovertemplate="<b>%{hovertext}</b><br>Población: %{marker.size:,}<extra></extra>",
)
fig.update_layout(legend_title=None)
fig.show()

{'sin indicadores': 5447, 'con indicadores': 1853}


Resumen:
- Las 7,300 localidades tienen coordenadas con el formato del descriptor.
- Ninguna cae fuera de la caja ni repite coordenadas con otra.
- La caja no descarta puntos en estados vecinos; para eso hace falta el Marco Geoestadístico de INEGI.
- En el mapa hay 1,853 localidades con indicadores y 5,447 sin ellos.

## 7. Población

Cuánta gente vive en cada localidad y qué tan concentrada está. Usa las 7,300 localidades.

In [24]:
pob.describe().round(1)

count      7300.0
mean        403.4
std       11671.2
min           1.0
25%           2.0
50%           4.0
75%           9.0
max      855563.0
Name: POBTOT, dtype: float64

Histograma de la población total en escala logarítmica: cada marca del eje multiplica por 10 a la anterior, para ver juntas localidades de unos cuantos y de cientos de miles de habitantes.

In [25]:
bordes = np.arange(0, 6.25, 0.25)
conteos, _ = np.histogram(np.log10(pob), bins=bordes)
histograma = pd.DataFrame(
    {
        "centro": bordes[:-1] + 0.125,
        "Localidades": conteos,
        "Habitantes": [f"{int(np.ceil(10**a)):,} a {int(np.ceil(10**b)) - 1:,}" for a, b in zip(bordes[:-1], bordes[1:])],
    }
)
fig = px.bar(
    histograma,
    x="centro",
    y="Localidades",
    hover_data={"centro": False, "Habitantes": True},
    title="Localidades según población total",
)
fig.update_traces(marker_color=AZUL)
fig.update_layout(
    height=420,
    bargap=0.05,
    barcornerradius=4,
    yaxis_title="Localidades",
    xaxis=dict(
        title="Población total (escala logarítmica)",
        tickvals=list(range(7)),
        ticktext=["1", "10", "100", "1,000", "10,000", "100,000", "1,000,000"],
    ),
)
fig.show()

Cuántas localidades, de la más poblada a la menos poblada, reúnen el 50 % y el 90 % de la población, y cuáles son las cinco más pobladas.

In [26]:
orden = pob.sort_values(ascending=False)
acumulado = orden.cumsum() / orden.sum() * 100
pd.Series({f"{p} %": int(np.searchsorted(acumulado.to_numpy(), p) + 1) for p in (50, 90)})

50 %     4
90 %    98
dtype: int64

In [27]:
locs.loc[orden.index[:5], ["NOM_MUN", "NOM_LOC", "POBTOT"]].assign(**{"% acumulado": acumulado.iloc[:5].round(1)})

,NOM_MUN,NOM_LOC,POBTOT,% acumulado
3352,Hermosillo,Hermosillo,855563,29.1
1269,Cajeme,Ciudad Obregón,329404,40.2
5291,Nogales,Heroica Nogales,261137,49.1
6105,San Luis Río Colorado,San Luis Río Colorado,176685,55.1
4983,Navojoa,Navojoa,120926,59.2


Localidades y población por tamaño de localidad, con los rangos del descriptor. También se revisa que la población de cada localidad caiga en el rango de su tamaño.

In [28]:
RANGOS_TAMLOC = {
    1: (1, 249), 2: (250, 499), 3: (500, 999), 4: (1_000, 2_499), 5: (2_500, 4_999),
    6: (5_000, 9_999), 7: (10_000, 14_999), 8: (15_000, 29_999), 9: (30_000, 49_999),
    10: (50_000, 99_999), 11: (100_000, 249_999), 12: (250_000, 499_999),
    13: (500_000, 999_999), 14: (1_000_000, np.inf),
}
tamloc = locs["TAMLOC"].astype(int)
minimo = tamloc.map(lambda t: RANGOS_TAMLOC[t][0])
maximo = tamloc.map(lambda t: RANGOS_TAMLOC[t][1])
print(f"Localidades con población fuera del rango de su tamaño: {(~pob.between(minimo, maximo)).sum()}")

por_tamloc = pd.DataFrame(
    {
        "rango": {t: f"{a:,} a {b:,}" if b < np.inf else f"{a:,} y más" for t, (a, b) in RANGOS_TAMLOC.items()},
        "localidades": tamloc.value_counts(),
        "población": pob.groupby(tamloc).sum(),
    }
).fillna({"localidades": 0, "población": 0}).astype({"localidades": int, "población": int})
por_tamloc["% localidades"] = (por_tamloc["localidades"] / len(locs) * 100).round(1)
por_tamloc["% población"] = (por_tamloc["población"] / pob.sum() * 100).round(1)
por_tamloc

Localidades con población fuera del rango de su tamaño: 0


,rango,localidades,población,% localidades,% población
1,1 a 249,6857,92430,93.9,3.1
2,250 a 499,178,63465,2.4,2.2
3,500 a 999,123,87177,1.7,3.0
4,"1,000 a 2,499",80,116099,1.1,3.9
5,"2,500 a 4,999",24,87135,0.3,3.0
6,"5,000 a 9,999",17,125156,0.2,4.3
7,"10,000 a 14,999",6,75086,0.1,2.5
8,"15,000 a 29,999",0,0,0.0,0.0
9,"30,000 a 49,999",6,216390,0.1,7.3
10,"50,000 a 99,999",3,220934,0.0,7.5


In [29]:
localidades_y_poblacion(
    por_tamloc.set_index("rango"), "Localidades y población por tamaño de localidad", "Habitantes", etiquetas=False
)

Resumen:
- La mitad de las localidades tiene 4 habitantes o menos.
- La más chica tiene 1 habitante: el ITER solo trae localidades habitadas.
- 4 localidades reúnen la mitad de la población: Hermosillo, Ciudad Obregón, Heroica Nogales y San Luis Río Colorado.
- 98 localidades reúnen el 90 % de la población.
- El 93.9 % de las localidades tiene menos de 250 habitantes. Ahí vive solo el 3.1 % de la población.
- La población de cada localidad siempre cae dentro del rango de su tamaño.

## 8. Indicadores de vulnerabilidad

Definición, denominador y distribución de los cinco indicadores. Usa las 1,853 localidades con indicadores; el porcentaje estatal sale de la fila del total del estado.

| Indicador | Columna | Definición en el descriptor | Denominador |
|---|---|---|---|
| 60 años y más | P_60YMAS | Personas de 60 a 130 años de edad. | Población total (POBTOT) |
| Con discapacidad | PCON_DISC | Personas que realizan con mucha dificultad o no pueden hacer al menos una de las siguientes actividades: ver, aun usando lentes; oír, aun usando aparato auditivo; caminar, subir o bajar; recordar o concentrarse; bañarse, vestirse o comer; hablar o comunicarse. | Población total (POBTOT) |
| Sin afiliación a servicios de salud | PSINDER | Total de personas que no están afiliadas a servicios médicos en ninguna institución pública o privada. | Población total (POBTOT) |
| Habla lengua indígena | P3YM_HLI | Personas de 3 a 130 años de edad que hablan alguna lengua indígena. | Población de 3 años y más (P_3YMAS) |
| Viviendas sin auto ni moto | VPH_NDACMM | Viviendas particulares habitadas que no tienen automóvil o camioneta ni motocicleta o motoneta. | Viviendas particulares habitadas con características (VIVPARH_CV) |

Según el descriptor, la población total incluye una estimación de quienes viven en viviendas sin información de sus ocupantes. Las viviendas sin auto ni moto y su denominador cubren las mismas viviendas.

In [30]:
completas = locs[disponibilidad.eq("indicadores completos")]
valores = completas[CONTEOS].apply(a_numero)
porcentajes = pd.DataFrame({v: valores[v] / valores[d] * 100 for v, d in INDICADORES.items()})
porcentajes.describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
P_60YMAS,1853.0,18.4,15.0,0.0,9.5,15.3,23.6,100.0
PCON_DISC,1853.0,6.6,7.1,0.0,0.0,5.4,9.2,66.7
PSINDER,1853.0,23.5,20.9,0.0,8.7,18.4,32.6,100.0
P3YM_HLI,1853.0,8.0,18.1,0.0,0.0,0.0,6.5,100.0
VPH_NDACMM,1850.0,41.0,24.4,0.0,25.0,40.0,56.3,100.0


Denominadores en cero, porcentajes fuera de 0 a 100 y porcentaje en la fila del total del estado.

In [31]:
fila_estatal = df[tipo.eq("total estatal")].iloc[0]
pd.DataFrame(
    {
        "denominador": INDICADORES,
        "denominador en cero": {v: valores[d].eq(0).sum() for v, d in INDICADORES.items()},
        "fuera de 0 a 100": {v: (~porcentajes[v].dropna().between(0, 100)).sum() for v in INDICADORES},
        "% estatal": {v: round(int(fila_estatal[v]) / int(fila_estatal[d]) * 100, 1) for v, d in INDICADORES.items()},
    }
)

,denominador,denominador en cero,fuera de 0 a 100,% estatal
P_60YMAS,POBTOT,0,0,12.2
PCON_DISC,POBTOT,0,0,4.9
PSINDER,POBTOT,0,0,18.6
P3YM_HLI,P_3YMAS,0,0,2.2
VPH_NDACMM,VIVPARH_CV,3,0,32.8


Los tamaños de localidad se juntan en 5 grupos. El corte de 2,500 habitantes es el que usa INEGI para separar localidades rurales y urbanas.

El porcentaje de cada grupo se calcula igual que el estatal: se suman las personas (o viviendas) con la condición en todas las localidades del grupo y se divide entre la suma de su denominador. Solo incluye las 1,853 localidades con indicadores. La última fila suma todas esas localidades.

In [32]:
GRUPOS_TAMLOC = {
    1: "1 a 249", 2: "250 a 999", 3: "250 a 999", 4: "1,000 a 2,499",
    5: "2,500 a 14,999", 6: "2,500 a 14,999", 7: "2,500 a 14,999",
    8: "15,000 y más", 9: "15,000 y más", 10: "15,000 y más", 11: "15,000 y más",
    12: "15,000 y más", 13: "15,000 y más", 14: "15,000 y más",
}
ORDEN_TAMANO = list(dict.fromkeys(GRUPOS_TAMLOC.values()))
TOTAL = "Total con indicadores"
UNIDAD = {v: "viviendas" if CONSERVAR.at[v, "tipo"] == "conteo de viviendas" else "personas" for v in INDICADORES}

tamano = completas["TAMLOC"].astype(int).map(GRUPOS_TAMLOC).rename("habitantes")
sumas = valores.groupby(tamano).sum().reindex(ORDEN_TAMANO)
sumas.loc[TOTAL] = valores.sum()
localidades_grupo = tamano.value_counts().reindex(ORDEN_TAMANO)
localidades_grupo[TOTAL] = len(completas)

por_tamano = pd.DataFrame(
    {
        "localidades": localidades_grupo,
        "habitantes": sumas["POBTOT"],
        "% de la población del estado": (sumas["POBTOT"] / pobtot[tipo.eq("total estatal")].sum() * 100).round(1),
    }
)
for v, d in INDICADORES.items():
    por_tamano[f"{NOMBRES[v]} (%)"] = (sumas[v] / sumas[d] * 100).round(1)
    por_tamano[f"{NOMBRES[v]} ({UNIDAD[v]})"] = sumas[v]
por_tamano.rename_axis("habitantes de la localidad")

,localidades,habitantes,% de la población del estado,60 años y más (%),60 años y más (personas),Con discapacidad (%),Con discapacidad (personas),Sin afiliación a servicios de salud (%),Sin afiliación a servicios de salud (personas),Habla lengua indígena (%),Habla lengua indígena (personas),Viviendas sin auto ni moto (%),Viviendas sin auto ni moto (viviendas)
habitantes de la localidad,,,,,,,,,,,,,
1 a 249,1417,74001,2.5,17.7,13127,6.9,5104,21.5,15938,10.5,7393,41.5,9269
250 a 999,295,148210,5.0,15.3,22606,6.5,9666,20.7,30696,11.2,15832,43.5,17937
"1,000 a 2,499",79,114332,3.9,14.7,16827,6.5,7389,20.9,23932,7.3,7941,45.3,14234
"2,500 a 14,999",47,287377,9.8,13.4,38559,6.0,17287,20.5,58916,5.2,14300,40.4,32567
"15,000 y más",15,2298292,78.0,11.5,263689,4.6,105050,17.9,411446,0.7,15075,30.5,211086
Total con indicadores,1853,2922212,99.2,12.1,354808,4.9,144496,18.5,540928,2.2,60541,32.8,285093


Porcentaje de cada indicador por grupo de tamaño. El cursor muestra cuántas personas o viviendas son.

In [33]:
barras_tamano = pd.DataFrame(
    [
        {
            "Indicador": NOMBRES[v],
            "Habitantes": g,
            "Porcentaje": por_tamano.at[g, f"{NOMBRES[v]} (%)"],
            "Cantidad": por_tamano.at[g, f"{NOMBRES[v]} ({UNIDAD[v]})"],
            "Unidad": UNIDAD[v],
        }
        for v in INDICADORES
        for g in ORDEN_TAMANO
    ]
)
fig = px.bar(
    barras_tamano,
    x="Indicador",
    y="Porcentaje",
    color="Habitantes",
    barmode="group",
    text="Porcentaje",
    custom_data=["Cantidad", "Unidad"],
    color_discrete_sequence=AZULES,
    category_orders={"Habitantes": ORDEN_TAMANO, "Indicador": list(NOMBRES.values())},
    title="Porcentaje de cada indicador por tamaño de localidad",
)
fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
    textfont_size=10,
    cliponaxis=False,
    hovertemplate="<b>%{x}</b><br>Localidades de %{fullData.name} habitantes: %{y:.1f} %"
    "<br>%{customdata[0]:,} %{customdata[1]}<extra></extra>",
)
fig.update_layout(
    height=480,
    xaxis_title=None,
    yaxis_title="Porcentaje del grupo",
    legend_title="Habitantes de la localidad",
    bargroupgap=0.05,
    barcornerradius=4,
)
fig.show()

Resumen:
- Ningún porcentaje pasa de 100.
- Solo las viviendas sin auto ni moto tienen denominador en cero, en 3 localidades que quedan sin porcentaje.
- En el estado, el 12.2 % de la población tiene 60 años o más y el 4.9 % tiene alguna discapacidad.
- El 18.6 % no tiene afiliación a servicios de salud y el 2.2 % habla lengua indígena.
- El 32.8 % de las viviendas no tiene auto ni moto.
- Las localidades de 15,000 habitantes o más reúnen el 78.0 % de la población del estado. Son solo 15.
- Las localidades de menos de 250 habitantes con indicadores reúnen el 2.5 %.
- Los cinco indicadores son más altos en las localidades chicas que en las de 15,000 habitantes o más.
- La lengua indígena es el caso más marcado: la habla el 10.5 % en las localidades de menos de 250 habitantes. En el grupo siguiente sube a 11.2 %.
- En las localidades de 15,000 habitantes o más, la lengua indígena baja a 0.7 %. Aun así, ahí viven 15,075 hablantes, casi tantos como los 15,832 del segundo grupo.
- Las personas mayores son el 17.7 % en las localidades de menos de 250 habitantes. En las de 15,000 o más bajan a 11.5 %.
- La discapacidad sigue el mismo patrón: va de 6.9 % a 4.6 %.
- La falta de afiliación va de 20.5 % a 21.5 % en los cuatro grupos más chicos. En las localidades de 15,000 o más baja a 17.9 %.
- Las viviendas sin auto ni moto van de 40.4 % a 45.3 % en los cuatro grupos más chicos. En las localidades de 15,000 o más bajan a 30.5 %.
- En número de personas, la mayoría está en las localidades de 15,000 habitantes o más. Por ejemplo, ahí viven 263,689 de las 354,808 personas de 60 años y más.
- En las localidades de menos de 250 habitantes, solo 1,417 tienen indicadores. En las más chicas la vulnerabilidad no se puede medir con el ITER.

## 9. Valores atípicos y localidades pequeñas

Localidades con valores muy alejados del resto y efecto del tamaño de la localidad en los porcentajes. Usa las 1,853 localidades con indicadores para los cinco porcentajes y las 7,300 localidades para las personas por vivienda habitada.

Los atípicos se buscan en porcentajes y en personas por vivienda, no en conteos: en un conteo, las ciudades grandes siempre salen como atípicas solo por su tamaño.

La regla: un valor es atípico si está a más de 1.5 veces el rango intercuartil por arriba del tercer cuartil o por debajo del primero. El primer cuartil deja abajo al 25 % de las localidades y el tercero al 75 %. El rango intercuartil es la distancia entre los dos.

Diagramas de caja de los cinco porcentajes y de las personas por vivienda habitada (población total entre viviendas habitadas), cada uno con su propia escala. En un diagrama de caja, la caja va del 25 % al 75 % de las localidades. La línea de en medio es la mediana: el valor que deja a la mitad de las localidades abajo y a la otra mitad arriba. Los puntos sueltos son los atípicos.

In [34]:
pob_c = pob[completas.index]
por_vivienda = pob / tvivhab


def cajas(datos, titulo, ejes_y, columnas=3):
    filas_fig = -(-len(datos) // columnas)
    fig = make_subplots(rows=filas_fig, cols=columnas, subplot_titles=list(datos), vertical_spacing=0.14)
    for i, (nombre, serie) in enumerate(datos.items()):
        fila, col = i // columnas + 1, i % columnas + 1
        fig.add_trace(
            go.Box(y=serie.dropna().round(2), name=nombre, marker_color=AZUL, boxpoints="outliers", showlegend=False),
            row=fila,
            col=col,
        )
        fig.update_yaxes(title_text=ejes_y[nombre], row=fila, col=col)
    fig.update_xaxes(showticklabels=False)
    fig.update_annotations(font_size=12)
    fig.update_layout(title=titulo, height=380 * filas_fig)
    fig.show()


cajas(
    {**{NOMBRES[v]: porcentajes[v] for v in INDICADORES}, "Personas por vivienda habitada": por_vivienda},
    "Porcentajes y personas por vivienda habitada",
    {**{NOMBRES[v]: "Porcentaje" for v in INDICADORES}, "Personas por vivienda habitada": "Personas por vivienda"},
)

Atípicos de cada variable y población de esas localidades contra la del resto.

In [35]:
def limites(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    return q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)


filas_atipicos = []
for nombre, serie in {**{v: porcentajes[v] for v in INDICADORES}, "personas por vivienda": por_vivienda}.items():
    serie = serie.dropna()
    bajo, alto = limites(serie)
    atipico = serie.lt(bajo) | serie.gt(alto)
    poblacion = pob[serie.index]
    filas_atipicos.append(
        {
            "variable": nombre,
            "con dato": len(serie),
            "límite inferior": round(bajo, 1),
            "límite superior": round(alto, 1),
            "atípicos bajos": serie.lt(bajo).sum(),
            "atípicos altos": serie.gt(alto).sum(),
            "mediana POBTOT atípicos": poblacion[atipico].median(),
            "mediana POBTOT resto": poblacion[~atipico].median(),
            "% población en atípicos": round(poblacion[atipico].sum() / poblacion.sum() * 100, 1),
        }
    )
pd.DataFrame(filas_atipicos).set_index("variable")

,con dato,límite inferior,límite superior,atípicos bajos,atípicos altos,mediana POBTOT atípicos,mediana POBTOT resto,% población en atípicos
variable,,,,,,,,
P_60YMAS,1853,-11.6,44.7,0,102,9.0,50.0,0.1
PCON_DISC,1853,-13.9,23.1,0,63,10.0,45.5,0.1
PSINDER,1853,-27.0,68.3,0,94,12.0,47.0,0.1
P3YM_HLI,1853,-9.8,16.3,0,278,104.5,36.0,3.4
VPH_NDACMM,1850,-22.0,103.3,0,0,NaN,42.0,0.0
personas por vivienda,7300,-2.0,7.4,0,119,11.0,3.0,0.3


Porcentajes extremos (0 % y 100 %) y población mediana de esas localidades contra la del resto.

In [36]:
pd.DataFrame(
    {
        v: {
            "en 0 %": porcentajes[v].eq(0).sum(),
            "en 100 %": porcentajes[v].eq(100).sum(),
            "mediana POBTOT en 0 o 100 %": pob_c[porcentajes[v].isin([0, 100])].median(),
            "mediana POBTOT del resto": pob_c[porcentajes[v].gt(0) & porcentajes[v].lt(100)].median(),
        }
        for v in INDICADORES
    }
).T.astype({"en 0 %": int, "en 100 %": int})

,en 0 %,en 100 %,mediana POBTOT en 0 o 100 %,mediana POBTOT del resto
P_60YMAS,174,7,12.0,59.5
PCON_DISC,488,0,12.0,107.0
PSINDER,211,17,11.0,64.0
P3YM_HLI,959,8,18.0,166.5
VPH_NDACMM,189,66,11.0,69.0


Localidades con indicadores por tamaño de población: cuántas son, qué parte de la población reúnen, qué parte tiene 60 años y más y viviendas sin auto ni moto en 0 % o 100 % y qué tan dispersos están sus valores (rango intercuartil de 60 años y más, en puntos porcentuales).

In [37]:
ELEGIDOS = ["P_60YMAS", "VPH_NDACMM"]
tramo = pd.cut(
    pob_c, [0, 19, 49, 99, 249, np.inf], labels=["menos de 20", "20 a 49", "50 a 99", "100 a 249", "250 y más"]
).rename("habitantes")
extremo = porcentajes[ELEGIDOS].isin([0, 100])
print(
    f"Localidades con indicadores y menos de 50 habitantes: {pob_c.lt(50).sum()}, "
    f"con el {pob_c[pob_c.lt(50)].sum() / pob.sum() * 100:.1f} % de la población del estado"
)
pd.DataFrame(
    {
        "localidades": tramo.value_counts(sort=False),
        "% de la población": (pob_c.groupby(tramo, observed=False).sum() / pob.sum() * 100).round(1),
        "% P_60YMAS en 0 o 100": (extremo["P_60YMAS"].groupby(tramo, observed=False).mean() * 100).round(1),
        "% VPH_NDACMM en 0 o 100": (extremo["VPH_NDACMM"].groupby(tramo, observed=False).mean() * 100).round(1),
        "rango intercuartil de P_60YMAS": porcentajes["P_60YMAS"].groupby(tramo, observed=False).quantile(0.75).sub(
            porcentajes["P_60YMAS"].groupby(tramo, observed=False).quantile(0.25)
        ).round(1),
    }
)

Localidades con indicadores y menos de 50 habitantes: 972, con el 0.6 % de la población del estado


,localidades,% de la población,% P_60YMAS en 0 o 100,% VPH_NDACMM en 0 o 100,rango intercuartil de P_60YMAS
habitantes,,,,,
menos de 20,637,0.2,23.1,34.4,26.7
20 a 49,335,0.4,8.4,9.0,20.3
50 a 99,178,0.4,2.8,1.7,14.9
100 a 249,267,1.5,0.4,1.1,10.5
250 y más,436,96.7,0.0,0.0,7.4


Resumen:
- En los porcentajes solo hay atípicos altos, porque el límite de abajo queda por debajo de 0.
- Las viviendas sin auto ni moto no tienen atípicos: su límite de arriba pasa de 100 %.
- En 60 años y más, discapacidad y falta de afiliación, los atípicos son localidades pequeñas: su población mediana va de 9 a 12 habitantes.
- En el resto de las localidades, la población mediana va de 45.5 a 50 habitantes.
- La lengua indígena es la excepción: sus atípicos son localidades más grandes, con una población mediana de 104.5 habitantes.
- Esas localidades reúnen el 3.4 % de la población. Parecen comunidades con población indígena, no un efecto del tamaño.
- En personas por vivienda, la regla pone el límite en 7.4. Ese límite también marca a familias grandes.
- Para señalar viviendas colectivas (penales, albergues o campamentos de trabajo), la sección 11 usa un umbral más alto: más de 10 personas por vivienda.
- Los porcentajes extremos (todo o nada) vienen de localidades pequeñas: su población mediana va de 11 a 18 habitantes.
- En las localidades de menos de 20 habitantes, el 23.1 % tiene un valor extremo en 60 años y más. En las de 250 o más, ninguna.
- La dispersión de 60 años y más (su rango intercuartil) baja de 26.7 a 7.4 puntos conforme crece la localidad.

Propuesta para el equipo (no aplicada):
- Usar los porcentajes solo en localidades de 50 habitantes o más. En las más chicas, una o dos personas mueven el porcentaje decenas de puntos.
- Las localidades con indicadores y menos de 50 habitantes reúnen solo el 0.6 % de la población.
- En el tablero, mostrar en esas localidades el conteo (por ejemplo, "3 de 8 personas") en lugar de colorearlas por porcentaje.
- Para mapas o listas, juntar las localidades pequeñas por municipio o por zona de acceso y calcular el porcentaje con la suma de sus conteos.
- Guardar el conteo y su denominador en los datos procesados, para poder cambiar el umbral después.
- No borrar los atípicos: casi todos vienen del tamaño de la localidad, y los de lengua indígena describen comunidades reales.

## 10. Mapas de vulnerabilidad

Un mapa por indicador, coloreado por porcentaje. Usa solo las localidades con indicadores y 50 habitantes o más, como propone la sección 9.

In [38]:
mapeables = completas.index[pob_c.ge(50)]
print(
    f"Localidades en los mapas: {len(mapeables):,}, "
    f"con el {pob[mapeables].sum() / pob.sum() * 100:.1f} % de la población del estado"
)

Localidades en los mapas: 881, con el 98.6 % de la población del estado


La escala de color va de 0 al valor que deja abajo al 95 % de las localidades del mapa. Las localidades por encima de ese valor llevan el color más oscuro, para que unos pocos valores altos no dejen todo el mapa en tonos claros.

In [39]:
def mapa_indicador(columna):
    datos = pd.DataFrame(
        {
            "etiqueta": cvegeo[mapeables] + " · " + locs.loc[mapeables, "NOM_MUN"],
            "Porcentaje": porcentajes.loc[mapeables, columna].round(1),
            "Población": pob[mapeables],
            "lat": lat[mapeables].round(5),
            "lon": lon[mapeables].round(5),
        }
    ).dropna()
    fig = px.scatter_map(
        datos,
        lat="lat",
        lon="lon",
        color="Porcentaje",
        range_color=(0, datos["Porcentaje"].quantile(0.95)),
        color_continuous_scale=AZULES,
        hover_name="etiqueta",
        custom_data=["Población"],
        center=CENTRO_SONORA,
        zoom=5.2,
        map_style="carto-positron",
        height=600,
        title=f"{NOMBRES[columna]}: porcentaje por localidad de 50 habitantes o más",
    )
    fig.update_traces(
        marker_size=8,
        hovertemplate="<b>%{hovertext}</b><br>Porcentaje: %{marker.color:.1f} %"
        "<br>Población: %{customdata[0]:,}<extra></extra>",
    )
    fig.update_layout(coloraxis_colorbar_title="%")
    fig.show()


for columna in INDICADORES:
    mapa_indicador(columna)

Dónde quedan las localidades con el color más oscuro de cada mapa: tope de la escala, cuántas localidades lo pasan y los municipios con más de ellas.

In [40]:
resumen_mapas = []
for columna in INDICADORES:
    valores_mapa = porcentajes.loc[mapeables, columna].dropna()
    tope = valores_mapa.quantile(0.95)
    altas = valores_mapa[valores_mapa.gt(tope)]
    municipios = locs.loc[altas.index, "NOM_MUN"].value_counts().head(3)
    resumen_mapas.append(
        {
            "indicador": NOMBRES[columna],
            "tope de la escala (%)": round(tope, 1),
            "localidades por encima": len(altas),
            "municipios con más localidades por encima": ", ".join(f"{m} ({n})" for m, n in municipios.items()),
        }
    )
pd.DataFrame(resumen_mapas).set_index("indicador")

,tope de la escala (%),localidades por encima,municipios con más localidades por encima
indicador,,,
60 años y más,32.3,44,"Alamos (8), Sahuaripa (4), Cumpas (3)"
Con discapacidad,14.6,44,"Alamos (11), Etchojoa (5), Navojoa (4)"
Sin afiliación a servicios de salud,48.9,44,"Caborca (6), Bavispe (4), Hermosillo (4)"
Habla lengua indígena,53.8,44,"Guaymas (15), Huatabampo (8), Alamos (6)"
Viviendas sin auto ni moto,75.0,41,"Etchojoa (9), Alamos (8), Guaymas (7)"


Resumen:
- Los mapas incluyen 881 localidades, con el 98.6 % de la población del estado.
- El tope de la escala va de 14.6 % en discapacidad a 75.0 % en viviendas sin auto ni moto.
- Alamos está entre los municipios con más localidades en el tono más oscuro en cuatro de los cinco mapas.
- Las localidades donde más se habla lengua indígena están sobre todo en Guaymas y Huatabampo.
- La falta de afiliación más alta aparece en Caborca, Bavispe y Hermosillo.
- Las viviendas sin auto ni moto pesan más en Etchojoa, Alamos y Guaymas.

## 11. Anomalías

Casos raros en las columnas que se conservan: viviendas colectivas, localidades no disponibles y nombres de localidad. Usa las 7,300 localidades.

Viviendas colectivas. Una vivienda colectiva es un penal, un albergue o un campamento de trabajo, donde viven muchas personas que no forman una familia. Se marcan las localidades con más de 10 personas por vivienda habitada. El umbral es más alto que el límite de 7.4 de la sección 9, para no marcar a familias grandes.

La marca sirve para dos cosas: decidir si su población cuenta como demanda de servicios de salud (un penal tiene su propio servicio médico) y no leer sus indicadores como los de un hogar. Resumen de esas localidades.

In [41]:
PALABRAS_USO = r"(?i)(readaptaci[oó]n|cefereso|invernadero|empaque|vi[ñn]edo|campo|agr[ií]cola|granja)"
uso = locs["NOM_LOC"].str.extract(PALABRAS_USO)[0].str.lower()
colectiva = por_vivienda.gt(10)
pd.Series(
    {
        "localidades con más de 10 personas por vivienda": colectiva.sum(),
        "con 1 vivienda habitada": (colectiva & tvivhab.eq(1)).sum(),
        "confidenciales (*)": (colectiva & disponibilidad.eq("confidencial (*)")).sum(),
        "con una palabra de uso en el nombre": (colectiva & uso.notna()).sum(),
        "población mediana": pob[colectiva].median(),
        "población total": pob[colectiva].sum(),
    },
    dtype=object,
)

localidades con más de 10 personas por vivienda      48
con 1 vivienda habitada                              34
confidenciales (*)                                   38
con una palabra de uso en el nombre                   9
población mediana                                  54.5
población total                                    8138
dtype: object

Caso representativo: entre las localidades confidenciales de una vivienda habitada, la que tiene la población más cercana a la mediana de ese grupo. Se muestra la clave, no el nombre.

In [42]:
tipicas = colectiva & tvivhab.eq(1) & disponibilidad.eq("confidencial (*)")
mediana_tipicas = pob[tipicas].median()
caso = (pob[tipicas] - mediana_tipicas).abs().idxmin()
print(f"Localidades en ese grupo: {tipicas.sum()}; población mediana: {mediana_tipicas:g}")
locs.loc[[caso], ["NOM_MUN", "TVIVHAB", "TAMLOC"]].assign(CVEGEO=cvegeo, POBTOT=pob, palabra=uso)

Localidades en ese grupo: 34; población mediana: 18.5


,NOM_MUN,TVIVHAB,TAMLOC,CVEGEO,POBTOT,palabra
2854,Etchojoa,1,1,260260798,18,NaN


Localidades confidenciales con 100 habitantes o más, con la palabra del nombre que indica su uso cuando la hay.

In [43]:
(
    locs.loc[disponibilidad.eq("confidencial (*)") & pob.ge(100), ["NOM_MUN", "TVIVHAB", "TAMLOC"]]
    .assign(CVEGEO=cvegeo, POBTOT=pob, palabra=uso)
    .sort_values("POBTOT", ascending=False)
)

,NOM_MUN,TVIVHAB,TAMLOC,CVEGEO,POBTOT,palabra
4350,Hermosillo,1,4,260304096,1767,cefereso
2659,Empalme,1,3,260250182,636,NaN
2668,Empalme,1,2,260250259,408,NaN
4222,Hermosillo,1,2,260303777,408,readaptación
4695,Imuris,1,2,260350309,372,invernadero
774,Bácum,1,2,260120415,352,NaN
1126,Caborca,1,2,260170817,256,NaN
6146,San Luis Río Colorado,1,1,260550164,206,empaque
4175,Hermosillo,1,1,260303656,139,NaN
4093,Hermosillo,2,1,260303327,128,viñedo


Población contra viviendas habitadas por localidad, en escala logarítmica (cada marca del eje multiplica por 10 a la anterior). En naranja, las de más de 10 personas por vivienda.

In [44]:
dispersion = pd.DataFrame({"CVEGEO": cvegeo[locs.index], "Viviendas habitadas": tvivhab, "Población": pob})
dispersion["Grupo"] = np.where(colectiva, "más de 10 personas por vivienda", "10 o menos")
fig = px.scatter(
    dispersion,
    x="Viviendas habitadas",
    y="Población",
    color="Grupo",
    log_x=True,
    log_y=True,
    hover_name="CVEGEO",
    color_discrete_map={"10 o menos": AZUL, "más de 10 personas por vivienda": NARANJA},
    title="Población contra viviendas habitadas por localidad",
)
fig.update_traces(
    marker_size=8,
    marker_line_width=1,
    marker_line_color="white",
    opacity=0.7,
    hovertemplate="<b>%{hovertext}</b><br>Viviendas habitadas: %{x}<br>Población: %{y:,}<extra></extra>",
)
fig.update_layout(height=480, legend_title=None)
fig.show()

Localidades no disponibles (N/D): población y viviendas habitadas.

In [45]:
no_disponibles = disponibilidad.eq("no disponible (N/D)")
pd.DataFrame({"población": pob[no_disponibles].describe(), "viviendas habitadas": tvivhab[no_disponibles].describe()}).round(1)

,población,viviendas habitadas
count,31.0,31.0
mean,21.2,6.0
std,16.7,4.6
min,3.0,3.0
25%,10.5,3.0
50%,16.0,4.0
75%,22.5,6.0
max,69.0,22.0


Nombres de localidad: genéricos, repetidos, con corchetes, con espacios de más o con posibles nombres de personas. Los nombres de personas solo se cuentan.

In [46]:
nombres = locs["NOM_LOC"]
PERSONA = r"(?i)\b(?:familia|hermanos|hnos|don|doña|ing|lic|dr|profr|profra)\b"
pd.Series(
    {
        "se llaman Ninguno": nombres.eq("Ninguno").sum(),
        "empiezan con Ninguno": nombres.str.startswith("Ninguno").sum(),
        "con corchetes": nombres.str.contains(r"\[").sum(),
        "con espacios dobles": nombres.str.contains("  ").sum(),
        "con espacios al inicio o al final": nombres.ne(nombres.str.strip()).sum(),
        "con Familia, Don, Doña, Hermanos o un título": nombres.str.contains(PERSONA).sum(),
    }
)

se llaman Ninguno                                62
empiezan con Ninguno                            259
con corchetes                                   406
con espacios dobles                               8
con espacios al inicio o al final                 0
con Familia, Don, Doña, Hermanos o un título     87
dtype: int64

Ejemplos de corchetes y de espacios dobles que no son nombres de personas.

In [47]:
dobles = nombres[nombres.str.contains("  ")]
print("Corchetes:", nombres[nombres.str.contains(r"\[(?:Club|Taller|Restaurante)")].head(2).tolist())
print("Espacios dobles:", dobles[dobles.str.contains(r"\(") & ~dobles.str.contains(r"\[")].head(2).tolist())

Corchetes: ['El Bajío [Club de Tiro]', 'Dávila [Taller]']
Espacios dobles: ['San Judas  (El Naranjal)', 'Agrícola COCI SA DE CV  (Pozo 18)']


Nombres repetidos dentro del mismo municipio.

In [48]:
repetidos = locs.duplicated(["MUN", "NOM_LOC"], keep=False)
print(
    f"Localidades con nombre repetido en su municipio: {repetidos.sum()}, "
    f"en {locs[repetidos].groupby(['MUN', 'NOM_LOC']).ngroups} nombres distintos"
)
locs[repetidos].groupby(["NOM_MUN", "NOM_LOC"]).size().sort_values(ascending=False).head(5)

Localidades con nombre repetido en su municipio: 605, en 246 nombres distintos


NOM_MUN     NOM_LOC      
Cajeme      Ninguno          21
Nogales     Ninguno          11
Hermosillo  San Francisco     8
Cajeme      San Antonio       7
Hermosillo  Ninguno           6
dtype: int64

Conteos negativos en las columnas que se conservan.

In [49]:
int((locs[CONTEOS].apply(a_numero) < 0).sum().sum())

0

Resumen:

| Anomalía | Casos | Ejemplo | Qué parece |
|---|---|---|---|
| Viviendas colectivas | 48 localidades con más de 10 personas por vivienda | Una localidad de Etchojoa con 1 vivienda y 18 personas | Regla del censo. Marcar y decidir con el equipo cómo contar su población. |
| Localidades confidenciales grandes | 12 con 100 habitantes o más | Un penal federal con 1,767 personas | Regla del censo: la confidencialidad depende de las viviendas, no de las personas. |
| Localidades no disponibles | 31 localidades | De 3 a 69 habitantes | Regla del censo. Preguntar si entran solo con su población. |
| Localidades "Ninguno" | 259 nombres empiezan así | 21 en Cajeme | Regla del censo: no tienen nombre oficial y el corchete da una referencia. |
| Nombres repetidos en el municipio | 605 localidades | San Francisco, 8 veces en Hermosillo | El nombre no sirve como llave. |
| Espacios dobles en el nombre | 8 nombres | `San Judas  (El Naranjal)` | Error de captura. |
| Posibles nombres de personas | 87 nombres | No se listan | Preguntar si se muestran en el tablero. |

- De las 48 localidades con viviendas colectivas, 38 son confidenciales. 34 de ellas tienen una sola vivienda.
- El caso típico es una localidad de una vivienda con unas 18 personas y sin pista de su uso en el nombre. Solo 9 dicen su uso: penal, invernadero, empaque, viñedo o campo.
- En total, esas 48 localidades suman 8,138 personas.
- No hay conteos negativos en las columnas que se conservan.

## 12. Notas para el diccionario y la limpieza

Confirmación de la regla del catálogo y notas para el diccionario, la limpieza y el equipo. La confirmación usa todas las filas del archivo.

El catálogo indica quitar las filas de totales y acumulados. Revisión de lo que queda al quitarlas.

In [50]:
sin_agregados = df[~df["LOC"].isin(["0000", "9998", "9999"])]
pd.Series(
    {
        "filas que quedan": len(sin_agregados),
        "claves únicas": cvegeo[sin_agregados.index].nunique(),
        "población que queda": pobtot[sin_agregados.index].sum(),
        "población del total estatal": pobtot[tipo.eq("total estatal")].sum(),
        "localidades de una y dos viviendas que quedan": a_numero(sin_agregados["TVIVHAB"]).le(2).sum(),
    }
)

filas que quedan                                    7300
claves únicas                                       7300
población que queda                              2944840
población del total estatal                      2944840
localidades de una y dos viviendas que quedan       5416
dtype: int64

Al quitar esas filas quedan las 7,300 localidades. Suman los 2,944,840 habitantes del estado, así que no se pierde ninguna.

| Columna | Cómo viene | Nota para la limpieza |
|---|---|---|
| ENTIDAD, MUN, LOC | texto de 2, 3 y 4 dígitos | Con ceros a la izquierda. Juntas forman la clave de 9 dígitos, única. |
| NOM_MUN, NOM_LOC | texto | NOM_LOC no sirve como llave: hay nombres repetidos en el mismo municipio y localidades "Ninguno". Algunos son nombres de personas y 8 tienen espacios dobles. |
| LATITUD, LONGITUD | texto en grados, minutos y segundos | Pasar a grados decimales, con la longitud negativa. |
| TAMLOC | texto de 1 o 2 caracteres | Rango de tamaño del descriptor, sin cero a la izquierda. Guardar como entero. |
| POBTOT, TVIVHAB | entero como texto | Con dato en todas las localidades. |
| P_3YMAS, VIVPARH_CV | entero como texto | Denominadores. "*" en las localidades confidenciales y "N/D" en las no disponibles. |
| P_60YMAS, PCON_DISC, PSINDER, P3YM_HLI, VPH_NDACMM | entero como texto | Indicadores. Las mismas marcas que sus denominadores. |

Columnas nuevas propuestas:
- Clave de 9 dígitos.
- Latitud y longitud en grados decimales.
- Disponibilidad: indicadores completos, confidencial o no disponible.
- Vivienda colectiva: más de 10 personas por vivienda habitada.
- Los cinco porcentajes, junto con su conteo y su denominador.
- Porcentaje estable: localidades de 50 habitantes o más, si el equipo aprueba la propuesta de la sección 9.

Tipos de dato: claves y nombres como texto; conteos como entero que admite vacíos (`Int64` en pandas); porcentajes y coordenadas como decimal; tamaño de localidad como entero.

Reglas de calidad posibles:
- Hay 7,300 localidades y su clave de 9 dígitos es única.
- La entidad siempre es 26 y hay 72 municipios.
- La población de las localidades suma lo mismo que el total del estado.
- En los conteos solo hay números, "*" o "N/D", y ningún número negativo.
- "*" solo aparece en localidades de una o dos viviendas habitadas.
- Ningún conteo pasa a su denominador.
- Los porcentajes van de 0 a 100.
- Las coordenadas caen dentro de la caja de Sonora y no se repiten.
- La población de cada localidad cae dentro del rango de su tamaño.

Limpieza:
- Quitar las filas de totales y acumulados.
- Pasar "*" y "N/D" a vacío y guardar aparte la disponibilidad de cada localidad.
- Pasar las coordenadas a grados decimales.
- Marcar las localidades con viviendas colectivas.
- Quitar los espacios dobles de los nombres en una columna aparte, sin tocar el nombre original.

Dudas para el equipo:
- ¿Las localidades confidenciales entran solo con su población, o se usan los acumulados para tener sus indicadores por municipio?
- ¿Las localidades no disponibles entran al análisis solo con su población?
- ¿La población de viviendas colectivas cuenta como demanda de servicios de salud? Los penales tienen su propio servicio médico.
- ¿Se aprueba usar los porcentajes solo en localidades de 50 habitantes o más?
- ¿La falta de afiliación se calcula sobre la población total, que incluye a quienes no dijeron si están afiliados?
- ¿Se muestran en el tablero los nombres de localidad que son nombres de personas?

## 13. Conclusiones

- El ITER ubica a toda la población: 7,300 localidades con coordenadas válidas y ninguna repetida. Las localidades de 15,000 habitantes o más reúnen el 78.0 % de la gente.
- Los indicadores de vulnerabilidad cubren al 99.23 % de la población. Faltan en las localidades de una o dos viviendas, que para el análisis de acceso cuentan con su población; sus indicadores solo existen sumados por municipio.
- Los cinco indicadores son más altos en las localidades chicas. La lengua indígena es el caso más marcado: la habla el 10.5 % en las localidades de menos de 250 habitantes. En las de 15,000 o más, solo el 0.7 %. Aun así, ahí viven 15,075 hablantes.
- Los porcentajes de localidades muy pequeñas cambian mucho con una o dos personas. Si se usan solo con 50 habitantes o más, siguen cubriendo al 98.6 % de la población.
- Hay 48 localidades con viviendas colectivas que conviene marcar, y decidir con el equipo cómo contar su población.